In [8]:
%pip install -q transformer_lens
%pip install -q circuitsvis




^C
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from transformer_lens.model_bridge import TransformerBridge
import circuitsvis as cv
from IPython.display import display

c:\Users\Fnu Aymen\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
model = TransformerBridge.boot_transformers("gpt2")

print("Number of layers:", model.cfg.n_layers)
print("Number of heads per layer:", model.cfg.n_heads)
print("Maximum context window:", model.cfg.n_ctx)

Loading weights: 100%|██████████| 148/148 [00:05<00:00, 27.12it/s]


Number of layers: 12
Number of heads per layer: 12
Maximum context window: 1024


In [ ]:
model_description_text = """## Loading Models

HookedTransformer comes loaded with >40 open source GPT-style models. You can load any of them in with `HookedTransformer.from_pretrained(MODEL_NAME)`. Each model is loaded into the consistent HookedTransformer architecture, designed to be clean, consistent and interpretability-friendly.

For this demo notebook we'll look at GPT-2 Small, an 80M parameter model. To try the model out, let's find the loss on this paragraph!"""

loss = model(model_description_text, return_type="loss")
print("Model loss:", loss)

Model loss: tensor(4.2514, grad_fn=<DivBackward0>)


In [ ]:
print(model.to_str_tokens("gpt2"))
print(model.to_str_tokens(["gpt2", "gpt2"]))
print(model.to_tokens("gpt2"))
print(model.to_string([50256, 70, 457, 17]))

['<|endoftext|>', 'g', 'pt', '2']
[['<|endoftext|>', 'g', 'pt', '2'], ['<|endoftext|>', 'g', 'pt', '2']]
tensor([[50256,    70,   457,    17]])
<|endoftext|>gpt2


Exercise - how many tokens does your model guess correctly?

In [ ]:
# Returns the logits for each token in the model
logits = model(model_description_text, return_type="logits")
print(logits.shape)
#Logits is a tensor of shape (1, 110, 50257)
# Every row in logits corresponds to a prediction for the next token
# To get the prediction for the next token, we take the argmax of the last dimension
# .squeeze() removes the batch dimension, and [:-1] removes the start token
prediction = logits.argmax(dim=-1).squeeze()[:-1]
tokens = model.to_tokens(model_description_text).squeeze()[1:]
print((prediction==tokens).sum().item()/prediction.shape[0])







torch.Size([1, 110, 50257])
0.29357798165137616


Exercise - visualise & inspect attention patterns


In [ ]:
model = TransformerBridge.boot_tl_legacy("NeelNanda/Attn_Only_2L512W_C4_Code")

text = "We think that powerful, significantly superhuman machine intelligence is more likely than not to be created this century. If current machine learning techniques were scaled up to this level, we think they would by default produce systems that are deceptive or manipulative, and that no solid plans are known for how to avoid this."

logits, cache = model.run_with_cache(text, remove_batch_dim=True)
str_tokens = model.to_str_tokens(text)

for layer in range(2):
    display(cv.attention.attention_patterns(
        tokens=str_tokens,
        attention=cache["pattern", layer],
    ))

c:\Users\Fnu Aymen\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Fnu Aymen\.cache\huggingface\hub\models--NeelNanda--gpt-neox-tokenizer-digits. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Fnu Aymen\AppData\Local\Programs\Python\Python311\Lib\site-pa

Exercise - write your own detectors

In [28]:
def current_attn_detector(cache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be current-token heads
    """

    heads = []
    for layer in range(2):
        for head in range(8):
            total = 0
            for row in range(62):
                if cache["pattern", layer][head][row].argmax(dim=-1)  == row:
                    total += 1
            if total/62 > 0.5:
                heads.append(f"{layer}.{head}")
    return(heads)


def prev_attn_detector(cache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be prev-token heads
    """
    heads = []
    for layer in range(2):
        for head in range(8):
            total = 0
            for row in range(62):
                if cache["pattern", layer][head][row].argmax(dim=-1)  == row-1:
                    total += 1
            if total/62 > 0.5:
                heads.append(f"{layer}.{head}")
    return(heads)

def first_attn_detector(cache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be first-token heads
    """
    heads = []
    for layer in range(2):
        for head in range(8):
            total = 0
            for row in range(62):
                if cache["pattern", layer][head][row].argmax(dim=-1)  == 0:
                    total += 1
            if total/62 > 0.5:
                heads.append(f"{layer}.{head}")
    return(heads)


print("Heads attending to current token  = ", ", ".join(current_attn_detector(cache)))
print("Heads attending to previous token = ", ", ".join(prev_attn_detector(cache)))
print("Heads attending to first token    = ", ", ".join(first_attn_detector(cache)))



Heads attending to current token  =  1.1, 1.4
Heads attending to previous token =  0.3
Heads attending to first token    =  0.1, 0.2, 0.4, 0.5, 0.6, 0.7, 1.0, 1.5, 1.6, 1.7
